# Practical Application: Customer Coupon Acceptance

**Dataset:** UCI In-Vehicle Coupon Recommendation  
**Objective:** Identify the driver and trip contexts associated with coupon acceptance, with emphasis on Bar coupons and an independent Coffee House investigation.

This notebook is organized as a reproducible statistical investigation. It reports denominators and segment sizes with every acceptance rate, distinguishes descriptive association from causal impact, and exports charts to `images/`.

**Outcome definition:** `Y = 1` means the driver accepted the coupon; `Y = 0` means it was not accepted.

In [ ]:
from pathlib import Path
import urllib.request
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, fisher_exact, norm

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
# Support execution either from repository root or from the notebooks directory.
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
IMAGE_DIR = ROOT / "images"
DATA_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "coupons.csv"
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00603/in-vehicle-coupon-recommendation.csv"

if not DATA_PATH.exists():
    print("Downloading the original UCI CSV...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df_raw = pd.read_csv(DATA_PATH)
print("Raw shape:", df_raw.shape)
display(df_raw.head())
print("\nColumn names:")
print(df_raw.columns.tolist())

## 1. Data understanding and data quality

Inspect the schema, duplicate rows and missingness before modifying the data. The `car` field is expected to be overwhelmingly missing and is not required for the segment questions, so it will be excluded. The visit-frequency fields have relatively small amounts of missing data and will be imputed with their mode after the missingness audit.

In [ ]:
print("Data types:")
display(df_raw.dtypes.rename("dtype").to_frame())

missing_audit = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2)
}).sort_values("missing_pct", ascending=False)
display(missing_audit)

print("Duplicate rows:", df_raw.duplicated().sum())
print("Outcome distribution:")
display(df_raw["Y"].value_counts(dropna=False).rename("count").to_frame())

## 2. Cleaning and preprocessing

Standardize column names and categorical text. The source uses the misspelling `passanger`; it is renamed to `passenger`. The `car` field is dropped because its missingness is too high to support reliable analysis. For the five visit-frequency variables, missing values are imputed with the modal category and the pre-imputation missing counts remain documented in `missing_audit`.

No rows are removed solely because of missing values in unrelated fields.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip()
df = df.rename(columns={"passanger": "passenger"})

# Drop the near-empty car field; it is not needed for this analysis.
if "car" in df.columns:
    df = df.drop(columns=["car"])

categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
for col in categorical_cols:
    df[col] = df[col].astype("string").str.strip()

frequency_cols = [
    "Bar", "CoffeeHouse", "CarryAway",
    "RestaurantLessThan20", "Restaurant20To50"
]
for col in frequency_cols:
    if col in df.columns:
        mode_values = df[col].mode(dropna=True)
        if not mode_values.empty:
            df[col] = df[col].fillna(mode_values.iloc[0])

# Normalize category spellings/whitespace without collapsing distinct source categories.
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].replace({"nan": pd.NA, "NaN": pd.NA, "": pd.NA})

# Ensure outcome is numeric and valid.
df["Y"] = pd.to_numeric(df["Y"], errors="coerce")
df = df[df["Y"].isin([0, 1])].copy()
df["Y"] = df["Y"].astype(int)

print("Cleaned shape:", df.shape)
print("Remaining missing values:")
display(df.isna().sum().loc[lambda x: x.gt(0)].to_frame("missing_count"))
print("Bar categories:", sorted(df["Bar"].dropna().unique().tolist()))
print("CoffeeHouse categories:", sorted(df["CoffeeHouse"].dropna().unique().tolist()))

## 3. Overall coupon acceptance

The overall acceptance rate is the share of all valid coupon presentations for which `Y=1`.

In [ ]:
overall_n = len(df)
overall_accepted = int(df["Y"].sum())
overall_rate = df["Y"].mean()

print(f"Coupon presentations: {overall_n:,}")
print(f"Accepted coupons: {overall_accepted:,}")
print(f"Overall acceptance rate: {overall_rate:.1%}")

overall_plot = df["Y"].value_counts(normalize=True).sort_index()
ax = overall_plot.plot(kind="bar", figsize=(6, 4), rot=0)
ax.set_xticks([0, 1], ["Not accepted (0)", "Accepted (1)"])
ax.set_ylabel("Proportion")
ax.set_title("Overall coupon acceptance")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "overall_acceptance.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Bar coupon investigation

All Bar coupon rates below use only rows where `coupon == "Bar"`. The frequency categories are interpreted as:
- **At most 3 visits/month:** `never`, `less1`, `1~3`
- **More than 3 visits/month:** `4~8`, `gt8`
- **More than once/month:** `1~3`, `4~8`, `gt8`

This makes the segment definitions explicit and reproducible.

In [ ]:
bar = df[df["coupon"].eq("Bar")].copy()
print("Bar coupon offers:", len(bar))
print(f"Bar coupon baseline acceptance: {bar['Y'].mean():.1%} ({bar['Y'].sum()}/{len(bar)})")

def segment_rate(data, mask, label):
    part = data.loc[mask].copy()
    return {
        "segment": label,
        "offers": len(part),
        "accepted": int(part["Y"].sum()),
        "acceptance_rate": part["Y"].mean() if len(part) else np.nan
    }

at_most_3 = bar["Bar"].isin(["never", "less1", "1~3"])
more_than_3 = bar["Bar"].isin(["4~8", "gt8"])
bar_frequency_results = pd.DataFrame([
    segment_rate(bar, at_most_3, "Bar visits: at most 3/month"),
    segment_rate(bar, more_than_3, "Bar visits: more than 3/month")
])
display(bar_frequency_results.style.format({"acceptance_rate":"{:.1%}"}))

### 4.1 Drivers visiting bars more than once per month and over age 25

The age field is categorical in the source. We map the provided age bands to a numeric lower-bound approximation for the specific threshold comparison. `50plus` is treated as 50 and `below21` as 20; the threshold is therefore based on the source's age-band labels.

In [ ]:
age_lower_bound = {
    "below21": 20, "21": 21, "26": 26, "31": 31,
    "36": 36, "41": 41, "46": 46, "50plus": 50
}
bar["age_lower_bound"] = bar["age"].astype(str).map(age_lower_bound)
visits_gt1 = bar["Bar"].isin(["1~3", "4~8", "gt8"])
age_over_25 = bar["age_lower_bound"].gt(25)

age_segment = segment_rate(
    bar, visits_gt1 & age_over_25,
    "Bar visits >1/month and age >25"
)
display(pd.DataFrame([age_segment]).style.format({"acceptance_rate":"{:.1%}"}))

### 4.2 Frequent bar visitors with passengers other than children, and not widowed

For this segment, "passengers other than children" means the driver has friends or a partner in the car. "Not widowed" excludes the `Widowed` marital-status category.

In [ ]:
non_child_companion = bar["passenger"].isin(["Friend(s)", "Partner"])
not_widowed = ~bar["maritalStatus"].eq("Widowed")
companion_segment = segment_rate(
    bar,
    visits_gt1 & non_child_companion & not_widowed,
    "Bar visits >1/month; friend/partner passenger; not widowed"
)
display(pd.DataFrame([companion_segment]).style.format({"acceptance_rate":"{:.1%}"}))

### 4.3 Combined segment: frequent bar visitors, no kids in car, non-farming occupation

The combined condition uses bar visits more than once per month, excludes `Kid(s)` as the passenger category, and excludes `Farming Fishing & Forestry` as the occupation. It retains alone, friend and partner travel contexts.

In [ ]:
no_kids_as_passenger = ~bar["passenger"].eq("Kid(s)")
non_farming = ~bar["occupation"].eq("Farming Fishing & Forestry")
combined_segment = segment_rate(
    bar,
    visits_gt1 & no_kids_as_passenger & non_farming,
    "Bar visits >1/month; no children as passengers; non-farming occupation"
)
display(pd.DataFrame([combined_segment]).style.format({"acceptance_rate":"{:.1%}"}))

### 4.4 Bar coupon segment summary and visualization

The table reports the exact denominator for each segment. Overlapping segments are not mutually exclusive and must not be added together.

In [ ]:
bar_segments = pd.DataFrame([
    segment_rate(bar, pd.Series(True, index=bar.index), "All Bar coupon offers"),
    segment_rate(bar, at_most_3, "Bar visits at most 3/month"),
    segment_rate(bar, more_than_3, "Bar visits more than 3/month"),
    age_segment,
    companion_segment,
    combined_segment
])
display(bar_segments.style.format({"acceptance_rate":"{:.1%}"}))

plot_data = bar_segments.dropna(subset=["acceptance_rate"]).sort_values("acceptance_rate")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(plot_data["segment"], plot_data["acceptance_rate"])
ax.set_xlim(0, 1)
ax.set_xlabel("Acceptance rate")
ax.set_title("Bar coupon acceptance by driver segment")
for y, (_, row) in enumerate(plot_data.iterrows()):
    ax.text(row["acceptance_rate"] + 0.01, y, f'{row["acceptance_rate"]:.1%} (n={int(row["offers"])})', va="center")
plt.tight_layout()
plt.savefig(IMAGE_DIR / "bar_segment_acceptance.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Independent investigation: Coffee House coupons

**Hypothesis:** Coffee House coupon acceptance is higher among drivers who visit coffee houses more frequently (at least 1~3 times per month) than among infrequent visitors (`never` or `less1`).

This is a two-sided comparison of independent proportions. The analysis reports group sizes and acceptance rates before the statistical test. It also explores time-of-day differences as descriptive context.

In [ ]:
coffee = df[df["coupon"].eq("Coffee House")].copy()
frequent_coffee = coffee["CoffeeHouse"].isin(["1~3", "4~8", "gt8"])
infrequent_coffee = coffee["CoffeeHouse"].isin(["never", "less1"])

coffee_groups = pd.DataFrame([
    segment_rate(coffee, frequent_coffee, "Coffee House visits >=1/month"),
    segment_rate(coffee, infrequent_coffee, "Coffee House visits <1/month")
])
display(coffee_groups.style.format({"acceptance_rate":"{:.1%}"}))

# Time-of-day comparison: morning (7AM/10AM) vs all other presented times.
morning = coffee["time"].isin(["7AM", "10AM"])
time_groups = pd.DataFrame([
    segment_rate(coffee, morning, "Morning offers (7AM/10AM)"),
    segment_rate(coffee, ~morning, "Other offer times")
])
display(time_groups.style.format({"acceptance_rate":"{:.1%}"}))

### 5.1 Two-proportion hypothesis test

Null hypothesis: the two group acceptance proportions are equal.  
Alternative hypothesis: the proportions differ.

The function below uses a pooled two-proportion z-test and reports a 95% confidence interval for the difference (group 1 minus group 2). A p-value is not evidence of causality.

In [ ]:
def two_proportion_test(x1, n1, x2, n2, label1, label2):
    if min(n1, n2) == 0:
        return {"comparison": f"{label1} vs {label2}", "p_value": np.nan}
    p1, p2 = x1/n1, x2/n2
    pooled = (x1+x2)/(n1+n2)
    se_pool = np.sqrt(pooled*(1-pooled)*(1/n1+1/n2))
    z = (p1-p2)/se_pool if se_pool else np.nan
    p_value = 2*(1-norm.cdf(abs(z))) if np.isfinite(z) else np.nan
    se_unpooled = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    diff = p1-p2
    ci_low, ci_high = diff-1.96*se_unpooled, diff+1.96*se_unpooled
    return {
        "comparison": f"{label1} minus {label2}",
        "rate_1": p1, "n_1": n1, "rate_2": p2, "n_2": n2,
        "difference": diff, "ci95_low": ci_low, "ci95_high": ci_high,
        "z_statistic": z, "p_value": p_value
    }

g1 = coffee.loc[frequent_coffee, "Y"]
g2 = coffee.loc[infrequent_coffee, "Y"]
coffee_test = two_proportion_test(
    int(g1.sum()), len(g1), int(g2.sum()), len(g2),
    "Frequent visitors", "Infrequent visitors"
)
display(pd.DataFrame([coffee_test]).style.format({
    "rate_1":"{:.1%}", "rate_2":"{:.1%}", "difference":"{:.1%}",
    "ci95_low":"{:.1%}", "ci95_high":"{:.1%}", "p_value":"{:.4f}"
}))

if pd.notna(coffee_test.get("p_value")):
    if coffee_test["p_value"] < 0.05:
        print("At the 5% level, the observed group difference is statistically significant.")
    else:
        print("At the 5% level, the analysis does not establish a statistically significant group difference.")

## 6. Business interpretation and mobile notification recommendations

Use the computed tables above to describe the highest-acceptance profiles. Do not infer a "most likely" profile from a high rate alone when the segment has a very small denominator. Always consider both acceptance rate and number of observations.

Suggested operational implications to validate with a campaign test:

- **Bar merchants:** prioritize audiences with demonstrated bar visitation frequency; use companion and age context only where segment sizes are adequate and privacy/marketing rules permit.
- **Coffee House merchants:** align offers with observed coffee-house visit frequency and test morning versus other time windows. Do not assume a morning effect unless the measured rates support it.
- **Push scheduling:** use the local time of the driving scenario as a starting hypothesis, but test timing, expiration and offer content using randomized holdouts.
- **Avoid over-targeting:** do not use sensitive or intrusive inferences. Segment on declared preferences and contextual relevance, with frequency caps and opt-out controls.

### Interpretation checklist

1. Quote the baseline rate and each segment's `n`.
2. Compare percentage-point differences, not only relative percentages.
3. Check confidence intervals and p-values before describing a difference as statistically significant.
4. Treat the survey as observational; validate recommendations with controlled campaign experiments.

In [ ]:
## 7. Export key summary tables

summary_tables = {
    "overall_acceptance": pd.DataFrame([{
        "offers": overall_n, "accepted": overall_accepted,
        "acceptance_rate": overall_rate
    }]),
    "bar_segments": bar_segments,
    "coffee_groups": coffee_groups,
    "coffee_time_groups": time_groups
}
for name, table in summary_tables.items():
    table.to_csv(ROOT / f"{name}.csv", index=False)

print("Saved summary CSVs to:", ROOT)
print("Saved charts to:", IMAGE_DIR)


## 8. Conclusion

Complete this section after running the notebook and reviewing the computed segment rates, sample sizes and hypothesis test. The key conclusion should identify the strongest supported Bar and Coffee House acceptance patterns, state the evidence and limitations, and translate the findings into testable notification strategies.

**Important:** the UCI data are survey responses to hypothetical driving scenarios. They do not establish that a push notification caused acceptance or that the same patterns will hold in a live campaign.